# Load a 3D NIfTI with itkwidgets (JupyterLite)

This notebook runs in the **Pyodide** kernel. Dependencies come from the docs checked into `PythonLightSuite/docs/`:

- **LightSuite** (`docs/LightSuite/`): atlas pipelines use NIfTI files; the Python helper `scripts/lsfm_atlas_inspect` loads volumes with **nibabel**, **numpy**, **matplotlib**, and **pandas**.
- **ITK-Wasm** (`docs/ITK-Wasm-main/`): WASM-friendly I/O via **itkwasm** / **itkwasm-image-io**. On Pyodide/emscripten only async readers exist — use **`await imread_async(...)`**, not `imread`.
- **itkwidgets** (`docs/itkwidgets-main`): interactive web viewers via `itkwidgets.view`; in JupyterLite use **`piplite`** and install **`imjoy-jupyterlab-extension`** in the site build (see [itkwidgets deployments](https://itkwidgets.readthedocs.io/en/latest/deployments.html)). Recent **`ngff-zarr` (≥0.10)** uses **OME-Zarr ≥0.5**, which requires **Zarr 3** and **`numcodecs>=0.14`**. **`numcodecs` has no PyPI wasm wheel**—Pyodide ships **0.13.x**—so upgrading Zarr with micropip fails. Here we **pin `ngff-zarr==0.9.1`** (still satisfies itkwidgets **≥0.8.7**) and **`zarr>=2,<3`** so `view()` stays on the Zarr **2** stack.

You can **download** a tiny sample NIfTI or **open a file** already in the JupyterLite directory (e.g. **`average_template_10.nii.gz`**). If you previously installed other versions in this tab, use **Restart kernel** and run from the top so **`reinstall=True`** can replace stale packages.


In [ ]:
import piplite

# Zarr 3 needs numcodecs>=0.14; PyPI has no Emscripten wheels for numcodecs, and Pyodide's
# bundled numcodecs is 0.13.x — so micropip cannot satisfy Zarr 3's deps. Pin ngff-zarr to
# 0.9.x (before 0.10's Zarr-3 / OME-0.5 writer path) + zarr 2.x — compatible with itkwidgets 1.0a55.
# reinstall=True: replace a newer ngff-zarr left over from an earlier session (e.g. 0.34).
await piplite.install(
    [
        "itkwidgets==1.0a55",
        "ngff-zarr[dask-image]==0.9.1",
        "zarr>=2.14,<3",
        "itkwasm-image-io",
    ],
    reinstall=True,
)


In [ ]:
from pyodide.http import pyfetch

# Small test volume from nibabel (BSD); suitable for smoke-testing I/O + viewer
NIFTI_URL = (
    "https://raw.githubusercontent.com/nipy/nibabel/master/"
    "nibabel/tests/data/anatomical.nii"
)


async def download(url: str, path: str) -> int:
    resp = await pyfetch(url)
    data = await resp.bytes()
    with open(path, "wb") as f:
        f.write(data)
    return len(data)


local_path = "anatomical.nii"
nbytes = await download(NIFTI_URL, local_path)
use_workspace_image = False  # viewer cell loads local_path when this is False
print(f"Downloaded {local_path} ({nbytes} bytes)")


### Load from the JupyterLite file tree

Put **`average_template_10.nii.gz`** where this notebook can see it:

- **Built site:** commit it under **`content/`** next to this notebook (same layout as in GitHub).
- **Interactive upload:** in JupyterLab, open the folder that contains **`01_load_nifti_itkwidgets.ipynb`** and upload there.

The next cell searches **`cwd`**, **`content/`**, and **`$HOME`** because the Pyodide kernel’s working directory is not always the notebook’s folder. It passes an **absolute path** into `imread_async` (relative paths can fail inside `itkwasm` even when the file exists).

In [ ]:
import os
from pathlib import Path

from itkwasm_image_io import imread_async

# Filename in the JupyterLite / Pyodide virtual filesystem (often under content/ after deploy).
WORKSPACE_NIFTI = "average_template_10.nii.gz"


def _resolve_workspace_nifti(name: str) -> Path:
    """Find the file: kernel cwd is not always the notebook folder in JupyterLite."""
    if Path(name).is_absolute():
        p = Path(name)
        if p.is_file():
            return p.resolve()
        raise FileNotFoundError(f"Not found: {name}")

    roots = (
        Path.cwd(),
        Path("content"),
        Path.cwd() / "content",
        Path(os.environ.get("HOME", "/home/pyodide")),
    )
    tried = []
    for root in roots:
        cand = root / name
        try:
            rp = str(cand.resolve())
        except OSError:
            rp = str(cand)
        tried.append(rp)
        if cand.is_file():
            return cand.resolve()

    plain = Path(name)
    tried.append(str(plain.resolve()))
    if plain.is_file():
        return plain.resolve()

    raise FileNotFoundError(
        f"{name} not found. Kernel cwd={Path.cwd()!s}. "
        "Upload in JupyterLab next to this notebook, or commit under content/ in git. "
        f"Checked: {tried}"
    )


_workspace_path = _resolve_workspace_nifti(WORKSPACE_NIFTI)
# itkwasm reads via open(path); absolute path avoids cwd mismatch inside itkwasm/pyodide.
_abs = str(_workspace_path.resolve())
image = await imread_async(_abs)
use_workspace_image = True  # viewer cell uses `image` from this cell instead of `local_path`
print(f"Loaded workspace file: {_workspace_path.name} ({_abs})")

In [ ]:
import warnings
from pathlib import Path

# Upstream itkwasm_image_io_emscripten still uses JsProxy.as_object_map(); Pyodide prefers as_py_json().
warnings.filterwarnings("ignore", message=r".*as_object_map.*", category=RuntimeWarning)

from itkwasm_image_io import imread_async
from itkwidgets import view

if not globals().get("use_workspace_image"):
    if "local_path" not in globals():
        raise RuntimeError("Run the download cell (sample NIfTI) or the workspace cell (your file) first.")
    _lp = Path(local_path)
    if _lp.is_file():
        image = await imread_async(str(_lp.resolve()))
    else:
        raise FileNotFoundError(f"{local_path} missing — run the download cell again.")

view(image, rotate=True)
